In [ ]:
# Cell 1：环境与配置（Kaggle 打开 GPU 和 Internet）
import os, sys, glob, time, random, json, math, hashlib, copy, gc, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from PIL import Image
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, log_loss, confusion_matrix

try:
    import timm
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'timm'])
    import timm
from timm.data.mixup import Mixup
from timm.loss import SoftTargetCrossEntropy

SEED = 42
TRAIN_DIR = '/kaggle/input/competitions/2026task1/dataset-for-task2/dataset-for-task2/train'
TEST_DIR = '/kaggle/input/competitions/2026task1/dataset-for-task2/dataset-for-task2/test'
WORK = Path('/kaggle/working')
CKPT_DIR = WORK / 'optimized_ckpts'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_SUB = None  # 如有官方 sample_submission.csv，可填完整路径
MODEL_NAME = 'convnext_small.fb_in22k_ft_in1k_384'
N_SPLITS = 3
SCREEN_EPOCHS = 8
FINAL_EPOCHS = 15
BATCH_SIZE = 16  # 显存充足可试 32；改动后会生成新的实验标识
NUM_WORKERS = 2  # GPU 利用率低、CPU 空闲时可试 4
PATIENCE = 4
MIN_EPOCHS = 8
LR = 2e-4
HEAD_LR = 1e-3
WEIGHT_DECAY = 0.02
LABEL_SMOOTHING = 0.05
REUSE_COMPLETED = True  # 只复用配置和数据一致、已完整结束的训练
COMPARE_384 = False  # 默认优先速度；True 会额外单折比较 384 分辨率
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda':
    raise RuntimeError('请在 Kaggle Settings → Accelerator 中启用 GPU。')
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
print('GPU:', torch.cuda.get_device_name(0), '| timm:', timm.__version__)



In [ ]:
# Cell 2：数据、固定划分和提交模板
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
def image_files(folder):
    return sorted(str(p) for p in Path(folder).iterdir()
                  if p.is_file() and p.suffix.lower() in EXTENSIONS)

if not Path(TRAIN_DIR).is_dir() or not Path(TEST_DIR).is_dir():
    raise FileNotFoundError('数据目录不存在，请按 Kaggle 的实际目录修改 TRAIN_DIR / TEST_DIR。')
classes = sorted(p.name for p in Path(TRAIN_DIR).iterdir() if p.is_dir())
train_files, train_labels = [], []
for label, name in enumerate(classes):
    files = image_files(Path(TRAIN_DIR) / name)
    train_files.extend(files)
    train_labels.extend([label] * len(files))
train_labels = np.asarray(train_labels, dtype=np.int64)
test_files = image_files(TEST_DIR)
if not train_files or not test_files or len(classes) < 2:
    raise ValueError('训练/测试集为空，或类别数量小于 2。')
counts = np.bincount(train_labels, minlength=len(classes))
if counts.min() < N_SPLITS:
    raise ValueError('某类别图片少于折数，请减小 N_SPLITS。')
folds = list(StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
             .split(train_files, train_labels))
manifest = [(str(Path(f).relative_to(TRAIN_DIR)), int(y),
             Path(f).stat().st_size, Path(f).stat().st_mtime_ns)
            for f, y in zip(train_files, train_labels)]
DATA_SIGNATURE = hashlib.sha256(json.dumps(manifest).encode()).hexdigest()
print('classes:', classes, '| train:', len(train_files), '| test:', len(test_files))
print('class counts:', dict(zip(classes, counts.tolist())))

if SAMPLE_SUB is None:
    candidates = sorted(Path(TRAIN_DIR).parents[2].rglob('*sample*submission*.csv'))
    if len(candidates) == 1:
        SAMPLE_SUB = str(candidates[0])
    elif len(candidates) > 1:
        print('检测到多个提交模板，请手动设置 SAMPLE_SUB；当前使用原代码 ID/Category 格式。')



In [ ]:
# Cell 3：增强和 DataLoader
pre_cfg = timm.models.get_pretrained_cfg(MODEL_NAME)
MEAN, STD = pre_cfg.mean, pre_cfg.std
CROP_PCT = pre_cfg.crop_pct or 0.875
INTERP = InterpolationMode.BICUBIC

def train_transform(size, aug):
    ops = [transforms.RandomResizedCrop(size, scale=(0.80, 1.0), interpolation=INTERP),
           transforms.RandomHorizontalFlip()]
    if aug == 'color':
        ops.append(transforms.ColorJitter(0.15, 0.15, 0.10, 0.02))
    ops += [transforms.ToTensor(), transforms.Normalize(MEAN, STD)]
    return transforms.Compose(ops)

def val_transform(size):
    return transforms.Compose([
        transforms.Resize(int(size / CROP_PCT), interpolation=INTERP),
        transforms.CenterCrop(size), transforms.ToTensor(),
        transforms.Normalize(MEAN, STD)])

class PlantDataset(Dataset):
    def __init__(self, files, labels, transform):
        self.files, self.labels, self.transform = files, labels, transform
    def __len__(self):
        return len(self.files)
    def __getitem__(self, index):
        with Image.open(self.files[index]) as image:
            x = self.transform(image.convert('RGB'))
        return x, int(self.labels[index]) if self.labels is not None else index

def seed_worker(worker_id):
    seed = torch.initial_seed() % 2**32
    random.seed(seed)
    np.random.seed(seed)

def make_loader(files, labels, transform, training=False, seed=SEED):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(PlantDataset(files, labels, transform),
                      batch_size=BATCH_SIZE if training else BATCH_SIZE * 2,
                      shuffle=training, num_workers=NUM_WORKERS,
                      pin_memory=True, persistent_workers=NUM_WORKERS > 0,
                      drop_last=False, worker_init_fn=seed_worker,
                      generator=generator)



In [ ]:
# Cell 4：EMA、推理和单折训练
class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.model = copy.deepcopy(model).eval()
        self.model.requires_grad_(False)
        self.decay, self.updates = decay, 0
    @torch.no_grad()
    def update(self, model):
        self.updates += 1
        decay = min(self.decay, (1 + self.updates) / (10 + self.updates))
        source = model.state_dict()
        for name, value in self.model.state_dict().items():
            if value.is_floating_point():
                value.mul_(decay).add_(source[name].detach(), alpha=1 - decay)
            else:
                value.copy_(source[name])

@torch.inference_mode()
def predict(model, loader, with_flip=False):
    model.eval()
    clean, flipped = [], []
    for x, _ in loader:
        x = x.to(DEVICE, non_blocking=True).contiguous(memory_format=torch.channels_last)
        with torch.amp.autocast('cuda'):
            p = model(x).float().softmax(1)
            if with_flip:
                q = model(torch.flip(x, dims=[3])).float().softmax(1)
        clean.append(p.cpu().numpy())
        if with_flip:
            flipped.append(((p + q) / 2).cpu().numpy())
    return np.concatenate(clean), np.concatenate(flipped) if with_flip else None

def metrics(y, p):
    return float(accuracy_score(y, p.argmax(1))), float(log_loss(y, p, labels=range(len(classes))))

def cpu_state(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

def load_checkpoint(path):
    # 只加载本代码生成的可信检查点；weights_only=False 兼容其中的元数据。
    return torch.load(path, map_location='cpu', weights_only=False)

def atomic_save(obj, path):
    temporary = Path(str(path) + '.tmp')
    torch.save(obj, temporary)
    os.replace(temporary, path)

def lr_factor(epoch, epochs):
    warmup = 2
    if epoch < warmup:
        return (epoch + 1) / warmup
    progress = (epoch - warmup) / max(epochs - 1 - warmup, 1)
    return 0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * progress))

def train_one(config, fold, epochs, stage):
    signature = dict(config=config, fold=fold, epochs=epochs, stage=stage,
                     seed=SEED, splits=N_SPLITS, data=DATA_SIGNATURE,
                     batch=BATCH_SIZE, lr=LR, head_lr=HEAD_LR, wd=WEIGHT_DECAY,
                     smoothing=LABEL_SMOOTHING, patience=PATIENCE,
                     min_epochs=MIN_EPOCHS, timm=timm.__version__, code_version=1)
    token = hashlib.sha256(json.dumps(signature, sort_keys=True).encode()).hexdigest()[:12]
    path = CKPT_DIR / f'{stage}_fold{fold}_{token}.pt'
    if REUSE_COMPLETED and path.exists():
        cached = load_checkpoint(path)
        if cached.get('completed') and cached.get('signature') == signature:
            print('复用已完成训练:', path.name, '| val:', cached['val_acc'])
            return path, cached['val_acc'], cached['val_loss']

    seed_everything(SEED + fold * 100)
    tr, va = folds[fold]
    tr_ld = make_loader([train_files[i] for i in tr], train_labels[tr],
                        train_transform(config['img'], config['aug']), True, SEED + fold)
    va_ld = make_loader([train_files[i] for i in va], train_labels[va], val_transform(config['img']))
    model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=len(classes),
                              drop_path_rate=0.1).to(DEVICE, memory_format=torch.channels_last)
    ema = ModelEMA(model)
    head_ids = {id(p) for p in model.get_classifier().parameters()}
    optimizer = torch.optim.AdamW([
        {'params': [p for p in model.parameters() if id(p) not in head_ids], 'lr': LR},
        {'params': [p for p in model.parameters() if id(p) in head_ids], 'lr': HEAD_LR}
    ], weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda ep: lr_factor(ep, epochs))
    scaler = torch.amp.GradScaler('cuda')
    hard_loss = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    soft_loss = SoftTargetCrossEntropy()
    mixup = Mixup(mixup_alpha=0.2, cutmix_alpha=0.0, prob=0.5, mode='elem',
                  label_smoothing=LABEL_SMOOTHING, num_classes=len(classes)) if config['mixup'] else None
    best_acc, best_loss, stale, best_epoch = -1.0, float('inf'), 0, 0
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        running_loss = torch.zeros((), device=DEVICE)
        examples = 0
        # 最后两轮关闭 Mixup；奇数 batch 使用硬标签以兼容 timm Mixup。
        for x, y in tr_ld:
            x = x.to(DEVICE, non_blocking=True).contiguous(memory_format=torch.channels_last)
            y = y.to(DEVICE, non_blocking=True)
            use_mix = mixup is not None and ep < epochs - 2 and len(y) % 2 == 0
            if use_mix:
                x, y = mixup(x, y)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'):
                logits = model(x)
                loss = soft_loss(logits, y) if use_mix else hard_loss(logits, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() >= old_scale:
                ema.update(model)  # 每次成功的优化器更新之后更新 EMA
            running_loss += loss.detach() * x.size(0)
            examples += x.size(0)
        scheduler.step()
        p, _ = predict(ema.model, va_ld)
        acc, val_loss = metrics(train_labels[va], p)
        print(f'{stage} fold{fold} ep{ep+1:02d}/{epochs} | '
              f'train {running_loss.item()/examples:.4f} | val {acc:.4f} | '
              f'loss {val_loss:.4f} | elapsed {(time.time()-t0)/60:.1f}min')
        improved = acc > best_acc or (acc == best_acc and val_loss < best_loss - 1e-5)
        if improved:
            best_acc, best_loss, stale, best_epoch = acc, val_loss, 0, ep + 1
            atomic_save(dict(signature=signature, state=cpu_state(ema.model),
                             config=config, classes=classes, model_name=MODEL_NAME,
                             val_acc=best_acc, val_loss=best_loss, best_epoch=best_epoch,
                             completed=False), path)
        else:
            stale += 1
        if ep + 1 >= MIN_EPOCHS and stale >= PATIENCE:
            print('Early stopping; best epoch:', best_epoch)
            break
    checkpoint = load_checkpoint(path)
    checkpoint['completed'] = True
    atomic_save(checkpoint, path)
    del model, ema, optimizer, scheduler, scaler, tr_ld, va_ld
    gc.collect()
    torch.cuda.empty_cache()
    return path, best_acc, best_loss



In [ ]:
# Cell 5：单折筛选（同一划分），自动选配置
CANDIDATES = [
    dict(img=224, aug='flip', mixup=False),
    dict(img=224, aug='color', mixup=False),
    dict(img=224, aug='color', mixup=True),
]
rows = []
for config in CANDIDATES:
    t0 = time.time()
    path, acc, loss = train_one(config, 0, SCREEN_EPOCHS, 'screen')
    rows.append(dict(**config, val_acc=acc, val_loss=loss,
                     minutes=(time.time()-t0)/60, checkpoint=str(path)))
    pd.DataFrame(rows).to_csv(WORK / 'screen_results.csv', index=False)
best = sorted(rows, key=lambda r: (-r['val_acc'], r['val_loss']))[0]
SELECTED = {k: best[k] for k in ('img', 'aug', 'mixup')}
if COMPARE_384:
    config = dict(SELECTED, img=384)
    t0 = time.time()
    path, acc, loss = train_one(config, 0, SCREEN_EPOCHS, 'screen')
    rows.append(dict(**config, val_acc=acc, val_loss=loss,
                     minutes=(time.time()-t0)/60, checkpoint=str(path)))
    best = sorted(rows, key=lambda r: (-r['val_acc'], r['val_loss']))[0]
    SELECTED = {k: best[k] for k in ('img', 'aug', 'mixup')}
pd.DataFrame(rows).to_csv(WORK / 'screen_results.csv', index=False)
print(pd.DataFrame(rows).drop(columns='checkpoint').to_string(index=False))
print('Selected:', SELECTED)



In [ ]:
# Cell 6：最佳配置三折训练
final_paths, final_rows = [], []
for fold in range(N_SPLITS):
    path, acc, loss = train_one(SELECTED, fold, FINAL_EPOCHS, 'final')
    final_paths.append(path)  # 仅使用本次选定的检查点，避免混入旧模型
    final_rows.append(dict(fold=fold, val_acc=acc, val_loss=loss, checkpoint=str(path)))
    pd.DataFrame(final_rows).to_csv(WORK / 'final_results.csv', index=False)
print('Final fold mean:', np.mean([r['val_acc'] for r in final_rows]))



In [ ]:
# Cell 7：验证 TTA、保存 OOF 和混淆矩阵
def model_from_checkpoint(path):
    checkpoint = load_checkpoint(path)
    if checkpoint['classes'] != classes:
        raise ValueError('检查点类别顺序不一致。')
    model = timm.create_model(checkpoint['model_name'], pretrained=False,
                              num_classes=len(classes), drop_path_rate=0.1)
    model.load_state_dict(checkpoint['state'])
    model = model.to(DEVICE, memory_format=torch.channels_last).eval()
    return model, checkpoint

oof_clean = np.zeros((len(train_files), len(classes)), dtype=np.float32)
oof_flip = np.zeros_like(oof_clean)
for fold, path in enumerate(final_paths):
    _, va = folds[fold]
    model, checkpoint = model_from_checkpoint(path)
    loader = make_loader([train_files[i] for i in va], train_labels[va],
                         val_transform(checkpoint['config']['img']))
    oof_clean[va], oof_flip[va] = predict(model, loader, with_flip=True)
    del model, loader
    torch.cuda.empty_cache()
acc_clean, loss_clean = metrics(train_labels, oof_clean)
acc_flip, loss_flip = metrics(train_labels, oof_flip)
USE_TTA = acc_flip > acc_clean or (acc_flip == acc_clean and loss_flip < loss_clean)
oof = oof_flip if USE_TTA else oof_clean
print(f'OOF clean: acc={acc_clean:.4f}, loss={loss_clean:.4f}')
print(f'OOF flip:  acc={acc_flip:.4f}, loss={loss_flip:.4f}; use TTA={USE_TTA}')
np.savez_compressed(WORK / 'oof_predictions.npz', clean=oof_clean, flip=oof_flip,
                    labels=train_labels, classes=np.array(classes), files=np.array(train_files))
pd.DataFrame(dict(file=train_files, true=[classes[i] for i in train_labels],
                  pred=[classes[i] for i in oof.argmax(1)], confidence=oof.max(1)))\
    .to_csv(WORK / 'oof_errors.csv', index=False)
import matplotlib.pyplot as plt
cm = confusion_matrix(train_labels, oof.argmax(1), labels=range(len(classes)))
pd.DataFrame(cm, index=classes, columns=classes).to_csv(WORK / 'confusion_matrix.csv')
fig, ax = plt.subplots(figsize=(max(6, len(classes)*0.6), max(5, len(classes)*0.6)))
image = ax.imshow(cm, cmap='Blues')
fig.colorbar(image, ax=ax)
ax.set(xticks=range(len(classes)), yticks=range(len(classes)),
       xticklabels=classes, yticklabels=classes, xlabel='Predicted', ylabel='True')
plt.setp(ax.get_xticklabels(), rotation=60, ha='right')
if len(classes) <= 15:
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > cm.max()/2 else 'black')
fig.tight_layout()
fig.savefig(WORK / 'confusion_matrix.png', dpi=150)
plt.show()
plt.close(fig)



In [ ]:
# Cell 8：三折等权集成、生成提交文件
test_probs = np.zeros((len(test_files), len(classes)), dtype=np.float32)
for path in final_paths:
    model, checkpoint = model_from_checkpoint(path)
    loader = make_loader(test_files, None, val_transform(checkpoint['config']['img']))
    clean, flipped = predict(model, loader, with_flip=USE_TTA)
    test_probs += (flipped if USE_TTA else clean) / len(final_paths)
    del model, loader
    torch.cuda.empty_cache()
names = [Path(f).name for f in test_files]
pred_labels = [classes[i] for i in test_probs.argmax(1)]
if SAMPLE_SUB:
    sub = pd.read_csv(SAMPLE_SUB, dtype=str)
    if len(sub.columns) != 2:
        raise ValueError('当前代码适用于 ID＋类别两列的分类提交，请核对官方模板。')
    id_col, label_col = sub.columns
    mapping = dict(zip(names, pred_labels))
    if sub[id_col].isin(mapping).all() and len(sub) == len(test_files) and sub[id_col].is_unique:
        sub[label_col] = sub[id_col].map(mapping)
    else:
        stem_names = [Path(name).stem for name in names]
        mapping = dict(zip(stem_names, pred_labels))
        if len(mapping) != len(test_files) or len(sub) != len(test_files) or not sub[id_col].is_unique or not sub[id_col].isin(mapping).all():
            raise ValueError('模板 ID 与测试文件名/去扩展名后的文件名无法一一对应，请核对后提交。')
        sub[label_col] = sub[id_col].map(mapping)
    print('使用官方模板顺序；标签沿用原代码中的文件夹名称，请确认符合比赛规定。')
else:
    sub = pd.DataFrame({'ID': names, 'Category': pred_labels})
    print('未指定官方模板，沿用原代码 ID/Category 和文件夹名称格式。')
submission_path = WORK / 'submission-for-task2.csv'
sub.to_csv(submission_path, index=False)
np.save(WORK / 'test_probabilities.npy', test_probs)
(WORK / 'selected_config.json').write_text(json.dumps(
    dict(config=SELECTED, use_tta=bool(USE_TTA), classes=classes,
         checkpoints=[str(p) for p in final_paths],
         oof_accuracy=acc_flip if USE_TTA else acc_clean), indent=2), encoding='utf-8')
print(sub.head())
print('类别分布:', pd.Series(pred_labels).value_counts().to_dict())
print('已生成:', submission_path, '| rows:', len(sub))


In [ ]:
# ===== 补充消融：固定第 0 折、224 分辨率、相同训练轮数 =====
# 依赖之前完整代码中的 train_one、transforms、WORK 等变量

required = ["train_one", "train_transform", "val_transform",
            "SCREEN_EPOCHS", "MODEL_NAME", "WORK"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        f"缺少变量 {missing}。请在之前训练的同一个 Notebook 会话中运行。"
    )

# 保存原增强函数，重复运行本单元也不会递归调用
if "_original_train_transform_ablation" not in globals():
    _original_train_transform_ablation = train_transform

def train_transform(size, aug):
    if aug == "none":
        # 保留统一尺寸和归一化，关闭随机裁剪、翻转、颜色变化
        return val_transform(size)
    return _original_train_transform_ablation(size, aug)

# 与原来的筛选实验保持一致，便于复用已完成的检查点
REUSE_COMPLETED = True

ablation_configs = [
    ("A：无随机增强", dict(img=224, aug="none",  mixup=False)),
    ("B：裁剪＋翻转", dict(img=224, aug="flip",  mixup=False)),
    ("C：增加颜色增强", dict(img=224, aug="color", mixup=False)),
    ("D：增加 Mixup", dict(img=224, aug="color", mixup=True)),
]

print("实际模型：", MODEL_NAME)
print("固定第 0 折，图片大小 224，训练轮数：", SCREEN_EPOCHS)
print("出现“复用已完成训练”表示该组不会重新训练。\n")

ablation_rows = []

for name, config in ablation_configs:
    print(f"\n===== {name} =====")
    start = time.time()

    # stage='screen' 与原代码一致，自动复用已有筛选结果
    path, acc, loss = train_one(
        config=config,
        fold=0,
        epochs=SCREEN_EPOCHS,
        stage="screen",
    )

    ablation_rows.append({
        "experiment": name,
        "aug": config["aug"],
        "mixup": config["mixup"],
        "val_acc": acc,
        "val_loss": loss,
        "此次耗时_分钟": round((time.time() - start) / 60, 2),
    })

    pd.DataFrame(ablation_rows).to_csv(
        WORK / "ablation_results.csv", index=False
    )

ablation_df = pd.DataFrame(ablation_rows)
baseline_acc = ablation_df.iloc[0]["val_acc"]

ablation_df["相对基线变化_百分点"] = (
    (ablation_df["val_acc"] - baseline_acc) * 100
).round(2)

ranked = ablation_df.sort_values(
    ["val_acc", "val_loss"],
    ascending=[False, True],
)

print("\n===== 消融结果 =====")
print(ablation_df.to_string(index=False))

best = ranked.iloc[0]
print("\n本次单折验证最佳：", best["experiment"])
print(f"验证准确率：{best['val_acc']:.2%}")
print("建议增强：", best["aug"], "| Mixup：", bool(best["mixup"]))
print("\n这里只筛选配置，不重新训练三折，也不覆盖现有提交文件。")

In [ ]:
# 只新增一次单折训练，其他条件与最佳增强配置保持一致
comparison = []

for size in [224, 384]:
    config = dict(img=size, aug="color", mixup=False)

    path, acc, loss = train_one(
        config=config,
        fold=0,
        epochs=SCREEN_EPOCHS,
        stage="screen",
    )

    comparison.append({
        "分辨率": size,
        "验证准确率": acc,
        "验证损失": loss,
    })

print(pd.DataFrame(comparison).to_string(index=False))